#### Simple Gen AI App using Langchain

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
# Langsmith Tracking
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAINGPT_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT")

In [2]:
## Data Ingestion from the website. We need to scrape the data
from langchain_community.document_loaders import WebBaseLoader

c:\Users\matcg\OneDrive\Documents\GitHub\LangChain_KrishNaik_Udemy\.venv312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [3]:
loader=WebBaseLoader("https://docs.langchain.com/langsmith/optimize-classifier")
loader

In [4]:
docs = loader.load()
docs

[Document(metadata={'source': 'https://docs.langchain.com/langsmith/optimize-classifier', 'title': 'Optimize a classifier - Docs by LangChain', 'language': 'en'}, page_content='Optimize a classifier - Docs by LangChainSkip to main contentDocs by LangChain home pageLangSmithSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationTutorialsOptimize a classifierGet startedObservabilityEvaluationPrompt engineeringDeploymentPlatform setupReferenceOverviewQuickstartConceptsCreate and update promptsCreate a promptManage promptsManage prompts programmaticallyPrompt template formatConfigure prompt settingsUse tools in a promptInclude multimodal content in a promptWrite your prompt with AIConnect to modelsTutorialsOptimize a classifierSync prompts with GitHubTest multi-turn conversationsOn this pageThe objectiveGetting startedSet up automationsUpdate the applicationSemantic search over examplesTutorialsOptimize a classifierCopy pageCopy pageThis tutorial walks through optimizing a cla

In [ ]:
#### Load Data --> Docs --> Divide documents into chunks--> text--> vectors --> Vector Embeddings --> Vector Store DB
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
documents = text_splitter.split_documents(docs)

In [6]:
documents

[Document(metadata={'source': 'https://docs.langchain.com/langsmith/optimize-classifier', 'title': 'Optimize a classifier - Docs by LangChain', 'language': 'en'}, page_content='Optimize a classifier - Docs by LangChainSkip to main contentDocs by LangChain home pageLangSmithSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationTutorialsOptimize a classifierGet startedObservabilityEvaluationPrompt engineeringDeploymentPlatform setupReferenceOverviewQuickstartConceptsCreate and update promptsCreate a promptManage promptsManage prompts programmaticallyPrompt template formatConfigure prompt settingsUse tools in a promptInclude multimodal content in a promptWrite your prompt with AIConnect to modelsTutorialsOptimize a classifierSync prompts with GitHubTest multi-turn conversationsOn this pageThe objectiveGetting startedSet up automationsUpdate the applicationSemantic search over examplesTutorialsOptimize a classifierCopy pageCopy pageThis tutorial walks through optimizing a cla

In [7]:
from langchain_openai import OpenAIEmbeddings
embeddings = OpenAIEmbeddings()

In [8]:
from langchain_community.vectorstores import FAISS
vectorstoredb = FAISS.from_documents(documents, embeddings)

In [9]:
vectorstoredb

In [12]:
## Query from vector store db
query="When interacting with it, we will generate the LangSmith run id ahead of time and pass that into this function"
result=vectorstoredb.similarity_search(query)
result[0].page_content

'We can then start to interact with it. When interacting with it, we will generate the LangSmith run id ahead of time and pass that into this function. We do this so we can attach feedback later on.\nHere’s how we can invoke the application:\nCopyfrom langsmith import uuid7\n\nrun_id = uuid7()\ntopic_classifier(\n    "fix bug in LCEL",\n    langsmith_extra={"run_id": run_id})\n\nHere’s how we can attach feedback after. We can collect feedback in two forms.\nFirst, we can collect “positive” feedback - this is for examples that the model got right.\nCopyls_client = Client()\nrun_id = uuid7()\ntopic_classifier(\n    "fix bug in LCEL",\n    langsmith_extra={"run_id": run_id})\nls_client.create_feedback(\n    run_id,\n    key="user-score",\n    score=1.0,\n)'

In [13]:
from langchain_openai import ChatOpenAI
llm=ChatOpenAI(model="gpt-4o")

In [ ]:
## Retrieval Chain, Document chain

from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

prompt=ChatPromptTemplate.from_template(
    """
    Answer the following question based on the context provided.
    <context>
    {context}
    </context>
    """
)
# O contexto seria o "paragrafo" para ele buscar a resposta, conforme similaridade. 
# Para que possamos passar o "paragrafo" correto, precisamos criar o document_chain, que é o responsável por pegar o resultado da similaridade e passar para o prompt.
document_chain=create_stuff_documents_chain(llm,prompt)
document_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\n    Answer the following question based on the context provided.\n    <context>\n    {context}\n    </context>\n    '), additional_kwargs={})])
| ChatOpenAI(profile={'max_input_tokens': 128000, 'max_output_tokens': 16384, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True}, client=<openai.resources.chat.com

In [19]:
# Podemos ver que o ChatPromptTemplate + ChatOpenAI + StrOutputParser estao combinados todo dentro do chain!

In [ ]:
from langchain_core.documents import Document
document_chain.invoke(
    {
        "input":"When interacting with it, we will generate the LangSmith run id ahead of time and pass that into this function",
        "context": [Document(page_content="""The LangSmith run id is generated before interacting with the system. 
                             Next, we can focus on collecting feedback that corresponds to a “correction” to the generation.""")]
    }
)

# Passamos o contexto de forma manual.

'The LangSmith run id is generated before interacting with the system. What happens next according to the context provided?'

However, we want the documents to first come from the retriever we just set up. That way, we can use the retriever to dynamically select the most relevant documents and pass those in for a given question.

In [ ]:
### Input-->Retriever-->vectorstoredb

vectorstoredb

In [25]:
retriever = vectorstoredb.as_retriever()
from langchain_classic.chains import create_retrieval_chain
retriever_chain = create_retrieval_chain(retriever, document_chain)
# O retriever vai pegar o resultado da similaridade e passar para o document_chain, que por sua vez vai passar para o prompt como contexto, que vai gerar a resposta.

In [26]:
retriever_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001F0BF300890>, search_kwargs={}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\n    Answer the following question based on the context provided.\n    <context>\n    {context}\n    </context>\n    '), additional_kwargs={})])
   

In [27]:
## Get the response from the LLM
response = retriever_chain.invoke({
    "input":"When interacting with it, we will generate the LangSmith run id ahead of time and pass that into this function" 
    })
response

{'input': 'When interacting with it, we will generate the LangSmith run id ahead of time and pass that into this function',
 'context': [Document(id='8f166de4-f5bc-4611-a66c-87f7fb592c5a', metadata={'source': 'https://docs.langchain.com/langsmith/optimize-classifier', 'title': 'Optimize a classifier - Docs by LangChain', 'language': 'en'}, page_content='We can then start to interact with it. When interacting with it, we will generate the LangSmith run id ahead of time and pass that into this function. We do this so we can attach feedback later on.\nHere’s how we can invoke the application:\nCopyfrom langsmith import uuid7\n\nrun_id = uuid7()\ntopic_classifier(\n    "fix bug in LCEL",\n    langsmith_extra={"run_id": run_id})\n\nHere’s how we can attach feedback after. We can collect feedback in two forms.\nFirst, we can collect “positive” feedback - this is for examples that the model got right.\nCopyls_client = Client()\nrun_id = uuid7()\ntopic_classifier(\n    "fix bug in LCEL",\n    

In [30]:
response['answer']

'Based on the provided context, here are some key points and processes described in the text:\n\n1. **LangSmith Interaction**: The interaction with the LangSmith application involves generating a `run_id` using `uuid7` and passing it into a function (like `topic_classifier`). This allows for attaching feedback to that specific run later on.\n\n2. **Feedback Collection**: Feedback can be collected in two ways:\n   - **Positive Feedback**: If the model performs correctly, feedback with a score (e.g., `1.0`) can be attached to a run.\n   - **Corrections and Datasets**: For runs requiring corrections, a webhook can be utilized to add these runs to a dataset, where the corrected output is used as the ground truth instead of the original output.\n\n3. **Update Application with Datasets**: The code can be updated to download datasets, and a function (`create_example_string`) is provided to format examples from the dataset into strings. These strings can be used as part of the prompt for the c

In [32]:
response['context']

[Document(id='8f166de4-f5bc-4611-a66c-87f7fb592c5a', metadata={'source': 'https://docs.langchain.com/langsmith/optimize-classifier', 'title': 'Optimize a classifier - Docs by LangChain', 'language': 'en'}, page_content='We can then start to interact with it. When interacting with it, we will generate the LangSmith run id ahead of time and pass that into this function. We do this so we can attach feedback later on.\nHere’s how we can invoke the application:\nCopyfrom langsmith import uuid7\n\nrun_id = uuid7()\ntopic_classifier(\n    "fix bug in LCEL",\n    langsmith_extra={"run_id": run_id})\n\nHere’s how we can attach feedback after. We can collect feedback in two forms.\nFirst, we can collect “positive” feedback - this is for examples that the model got right.\nCopyls_client = Client()\nrun_id = uuid7()\ntopic_classifier(\n    "fix bug in LCEL",\n    langsmith_extra={"run_id": run_id})\nls_client.create_feedback(\n    run_id,\n    key="user-score",\n    score=1.0,\n)'),
 Document(id='

In [31]:
print(response['answer'])

Based on the provided context, here are some key points and processes described in the text:

1. **LangSmith Interaction**: The interaction with the LangSmith application involves generating a `run_id` using `uuid7` and passing it into a function (like `topic_classifier`). This allows for attaching feedback to that specific run later on.

2. **Feedback Collection**: Feedback can be collected in two ways:
   - **Positive Feedback**: If the model performs correctly, feedback with a score (e.g., `1.0`) can be attached to a run.
   - **Corrections and Datasets**: For runs requiring corrections, a webhook can be utilized to add these runs to a dataset, where the corrected output is used as the ground truth instead of the original output.

3. **Update Application with Datasets**: The code can be updated to download datasets, and a function (`create_example_string`) is provided to format examples from the dataset into strings. These strings can be used as part of the prompt for the classifier